# Knowledge Engineering: Autonomous Trailer Navigation Agent

**Course:** ICS 3202 · Practical Exercise 4 — Developing a Knowledge-Based Agent

**Group members:** Denley Kipkoech, Natasha Kemunto.

This notebook builds on Practical Exercise 3 (the self-driving trailer hauling cargo from Nairobi to Kisumu). The towns, highway links and distances from that exercise's directed graph are now stored as a **knowledge base**. The agent can be told new facts, and can be asked questions that it answers by inference. The structure follows the 10-step knowledge-engineering template used in class.

_Structure adapted from the class knowledge-engineering template (Zainabu Muti, via AllanVikiru/ArtificialIntelligence). Knowledge base built from our Practical Exercise 3 design: https://github.com/denleykl/PS-Agents/blob/main/design.md_


**Step 1: Identify the Task**

The task is to help a self-driving trailer decide **which route to take** between towns on Kenya's highway network, using only the facts in its knowledge base.

**Goal:** Deliver cargo from Nairobi to Kisumu (or between any two known towns) over the shortest total distance, and therefore with the least time and fuel.

**Step 2: Assemble Relevant Knowledge**

The system needs to know:

- Which towns exist
- Which towns are joined by a highway, and the road distance in km
- That a highway can be driven in both directions at the same cost

**Source:** the abstracted directed graph and distance table from Practical Exercise 3 (approximate road distances between towns). Two extra illustrative roads are added in Step 5 to show how the knowledge base grows.

**APIs / web scraping:** none were used. All knowledge was encoded by hand from Practical Exercise 3.

**Step 3: Define the Vocabulary**

**Constants (towns):** Nairobi, Thika, Murang'a, Nyeri, Nyahururu, Naivasha, Nakuru, Kericho, Eldoret, Kisumu

**Predicates** (represented in Python as dictionary entries and functions):

- `Road(a, b, km)`: a highway of `km` kilometres links town `a` and town `b`
- `Connected(a, b)`: a road links `a` and `b` directly
- `Route(start, goal, path, cost)`: `path` is the least-cost route from `start` to `goal`

**Rules:**

1. `Road(a, b, km)` ⇒ `Road(b, a, km)` (highways are two-way)
2. `Road(a, b, km)` ⇒ `Connected(a, b)`
3. `cost(path)` = sum of the `Road` distances along the path
4. The optimal route is the path with minimum cost (found by uniform-cost search, as in Practical Exercise 3)

**Step 4: Encode General Knowledge about the Domain**

The facts below come from Practical Exercise 3. Each road is stated once; rule 1 (two-way roads) is applied automatically by `tell_road`, which stores it in both directions.

In [1]:
import heapq
import copy
import re

# Facts from Practical Exercise 3: Road(a, b, km). Each link is listed once.
road_facts = [
    ("Nairobi", "Thika", 45),
    ("Nairobi", "Naivasha", 90),
    ("Thika", "Murang'a", 40),
    ("Murang'a", "Nyeri", 55),
    ("Nyeri", "Nyahururu", 95),
    ("Nyahururu", "Nakuru", 90),
    ("Naivasha", "Nakuru", 65),
    ("Nakuru", "Kericho", 105),
    ("Nakuru", "Eldoret", 155),
    ("Kericho", "Kisumu", 105),
    ("Eldoret", "Kisumu", 150),
]

# The knowledge base: kb[town] = {neighbouring_town: distance_km}
kb = {}


def normalise(name):
    """Lower-case a name and drop everything except letters, so "Murang'a" matches "muranga"."""
    return re.sub(r"[^a-z]", "", name.lower())


def resolve_town(name, graph=None):
    """Return the town's stored name if it is in the KB, otherwise None."""
    graph = kb if graph is None else graph
    key = normalise(name)
    if not key:
        return None
    for town in graph:
        if normalise(town) == key:
            return town
    return None


def clean_name(name, graph=None):
    """Use the stored spelling of a known town, or tidy up the capitalisation of a new one."""
    known = resolve_town(name, graph)
    if known:
        return known
    return " ".join(w[:1].upper() + w[1:] for w in name.strip().split())


def fmt_km(km):
    return int(km) if float(km) == int(km) else km


def tell_road(a, b, km, graph=None):
    """Add Road(a, b, km) and, by the two-way rule, Road(b, a, km)."""
    graph = kb if graph is None else graph
    a, b = clean_name(a, graph), clean_name(b, graph)
    if not a or not b:
        raise ValueError("Town names cannot be empty.")
    if normalise(a) == normalise(b):
        raise ValueError("A road must join two different towns.")
    try:
        km = float(km)
    except (TypeError, ValueError):
        raise ValueError("Distance must be a number.")
    if km <= 0:
        raise ValueError("Distance must be greater than zero.")
    graph.setdefault(a, {})[b] = km
    graph.setdefault(b, {})[a] = km
    return a, b, km


for a, b, km in road_facts:
    tell_road(a, b, km)

print(f"Knowledge base loaded: {len(kb)} towns, {len(road_facts)} roads (each stored in both directions).")

Knowledge base loaded: 10 towns, 11 roads (each stored in both directions).


**Step 5: Add New Facts to the Knowledge Base**

New facts can be added in two ways: directly in code (`add_new_fact`) and from user input (`add_road`). Press Enter at the prompt to skip adding a road.

The two example roads below are approximate and illustrative; they extend the network beyond the Practical Exercise 3 corridor.

In [2]:
def show_kb(graph=None):
    graph = kb if graph is None else graph
    print("\n--- Current Knowledge Base ---")
    for town in sorted(graph):
        links = ", ".join(f"{n} ({fmt_km(km)} km)" for n, km in sorted(graph[town].items()))
        print(f"{town:12} → {links}")
    print("------------------------------")


def add_new_fact(a, b, km, graph=None):
    """Add a new road fact directly in code (without user input)."""
    a, b, km = tell_road(a, b, km, graph)
    print(f"Added fact: Road({a}, {b}, {fmt_km(km)}) and Road({b}, {a}, {fmt_km(km)})")


# Example of adding facts in code (approximate distances)
add_new_fact("Kisumu", "Kakamega", 50)
add_new_fact("Eldoret", "Kitale", 73)


def add_road():
    """Add a new road to the KB from user input (press Enter to skip)."""
    a = input("Enter the first town (or press Enter to skip): ").strip()
    if not a:
        print("Skipped: no new road added.")
        return
    b = input("Enter the second town: ").strip()
    km = input(f"Distance between {a} and {b} in km: ").strip()
    try:
        add_new_fact(a, b, km)
    except ValueError as err:
        print(f"Could not add fact: {err}")


add_road()

# View the updated KB
show_kb()

Added fact: Road(Kisumu, Kakamega, 50) and Road(Kakamega, Kisumu, 50)
Added fact: Road(Eldoret, Kitale, 73) and Road(Kitale, Eldoret, 73)
Skipped: no new road added.

--- Current Knowledge Base ---
Eldoret      → Kisumu (150 km), Kitale (73 km), Nakuru (155 km)
Kakamega     → Kisumu (50 km)
Kericho      → Kisumu (105 km), Nakuru (105 km)
Kisumu       → Eldoret (150 km), Kakamega (50 km), Kericho (105 km)
Kitale       → Eldoret (73 km)
Murang'a     → Nyeri (55 km), Thika (40 km)
Nairobi      → Naivasha (90 km), Thika (45 km)
Naivasha     → Nairobi (90 km), Nakuru (65 km)
Nakuru       → Eldoret (155 km), Kericho (105 km), Naivasha (65 km), Nyahururu (90 km)
Nyahururu    → Nakuru (90 km), Nyeri (95 km)
Nyeri        → Murang'a (55 km), Nyahururu (95 km)
Thika        → Murang'a (40 km), Nairobi (45 km)
------------------------------


**Steps 6 & 7: Encode a Specific Problem Instance (Facts) & Pose Queries and Get Answers**

Step 6 records the current problem: where the trailer is and where it must go. Step 7 queries the knowledge base. The agent infers the least-cost route with uniform-cost search, always expanding the frontier town with the lowest cumulative distance. The first time the goal is removed from the frontier, its path is optimal.

In [3]:
# Step 6: the specific problem instance (press Enter for the defaults)
start_in = input("Enter the starting town (Enter for Nairobi): ").strip() or "Nairobi"
goal_in = input("Enter the destination town (Enter for Kisumu): ").strip() or "Kisumu"

In [4]:
# Step 7: the agent queries the knowledge base
def find_route(start, goal, graph=None):
    """Uniform-cost search over the KB. Returns a dict describing the answer."""
    graph = kb if graph is None else graph
    s, g = resolve_town(start, graph), resolve_town(goal, graph)
    for given, found in ((start, s), (goal, g)):
        if found is None:
            if not given.strip():
                return {"status": "unknown_town", "message": "No town name was given."}
            return {"status": "unknown_town",
                    "message": f"Unknown town: '{given.strip()}' is not in the knowledge base."}

    frontier = [(0, 0, s, [s])]       # (cost so far, tie-breaker, town, path)
    best = {s: 0}
    expanded = []
    tie = 1
    while frontier:
        cost, _, town, path = heapq.heappop(frontier)
        if cost > best[town]:
            continue                  # an older, more expensive entry for this town
        expanded.append((town, cost))
        if town == g:
            return {"status": "found", "path": path, "cost": cost, "expanded": expanded}
        for nxt, km in graph[town].items():
            new_cost = cost + km
            if new_cost < best.get(nxt, float("inf")):
                best[nxt] = new_cost
                heapq.heappush(frontier, (new_cost, tie, nxt, path + [nxt]))
                tie += 1
    return {"status": "no_route", "expanded": expanded,
            "message": f"No known route from {s} to {g} in the knowledge base."}


def print_route(result, show_trace=False):
    if result["status"] != "found":
        print("Agent decision:", result["message"])
        return
    if len(result["path"]) == 1:
        print(f"Agent decision: already at {result['path'][0]} (0 km).")
    else:
        print("Agent decision: take", " → ".join(result["path"]))
        print(f"Total distance: {fmt_km(result['cost'])} km")
    if show_trace:
        trace = ", ".join(f"{t} ({fmt_km(c)})" for t, c in result["expanded"])
        print("Towns expanded (cost so far):", trace)


result = find_route(start_in, goal_in)
print("\n=== Autonomous Trailer Route Planner ===")
print(f"Start: {start_in}    Destination: {goal_in}")
print_route(result, show_trace=True)


=== Autonomous Trailer Route Planner ===
Start: Nairobi    Destination: Kisumu
Agent decision: take Nairobi → Naivasha → Nakuru → Kericho → Kisumu
Total distance: 365 km
Towns expanded (cost so far): Nairobi (0), Thika (45), Murang'a (85), Naivasha (90), Nyeri (140), Nakuru (155), Nyahururu (235), Kericho (260), Eldoret (310), Kisumu (365)


**Step 8: Querying the Knowledge Base**

More ways to ask the knowledge base questions: whether two towns are directly connected, the distance of a road, a town's neighbours, and routes between different towns.

In [5]:
def ask_connected(a, b, graph=None):
    graph = kb if graph is None else graph
    a, b = resolve_town(a, graph), resolve_town(b, graph)
    return a is not None and b is not None and b in graph[a]


def ask_distance(a, b, graph=None):
    graph = kb if graph is None else graph
    a, b = resolve_town(a, graph), resolve_town(b, graph)
    if a is None or b is None or b not in graph[a]:
        return None
    return fmt_km(graph[a][b])


def ask_neighbours(a, graph=None):
    graph = kb if graph is None else graph
    a = resolve_town(a, graph)
    return None if a is None else sorted(graph[a])


# Ask questions to get answers based on known facts
print("Connected(Nairobi, Thika)?   ", ask_connected("Nairobi", "Thika"))
print("Connected(Nairobi, Kisumu)?  ", ask_connected("Nairobi", "Kisumu"))
print("Distance Nakuru - Kericho:   ", ask_distance("Nakuru", "Kericho"), "km")
print("Neighbours of Nakuru:        ", ask_neighbours("Nakuru"))

print()
for s, g in [("Nairobi", "Kisumu"), ("Nairobi", "Eldoret"), ("Thika", "Nyahururu"), ("Kisumu", "Nairobi")]:
    print(f"Route {s} → {g}:")
    print_route(find_route(s, g))
    print()

Connected(Nairobi, Thika)?    True
Connected(Nairobi, Kisumu)?   False
Distance Nakuru - Kericho:    105 km
Neighbours of Nakuru:         ['Eldoret', 'Kericho', 'Naivasha', 'Nyahururu']

Route Nairobi → Kisumu:
Agent decision: take Nairobi → Naivasha → Nakuru → Kericho → Kisumu
Total distance: 365 km

Route Nairobi → Eldoret:
Agent decision: take Nairobi → Naivasha → Nakuru → Eldoret
Total distance: 310 km

Route Thika → Nyahururu:
Agent decision: take Thika → Murang'a → Nyeri → Nyahururu
Total distance: 190 km

Route Kisumu → Nairobi:
Agent decision: take Kisumu → Kericho → Nakuru → Naivasha → Nairobi
Total distance: 365 km



**Step 9: Debugging the Knowledge Base** 

Test unknown, faulty and unusual inputs to see how the agent reacts. The tests run on a **copy** of the knowledge base, so the extra test data (a Mombasa–Voi road that is not connected to the rest of the network) does not affect the real one.

In [7]:
debug_kb = copy.deepcopy(kb)
add_new_fact("Mombasa", "Voi", 150, debug_kb)     # an isolated pair of towns, used to test "no route"

test_cases = [
    ("Nairobi", "Kisumu"),      # normal query
    ("nairobi", "  KISUMU "),   # different capitalisation and spacing
    ("muranga", "nyeri"),       # spelling without the apostrophe
    ("Nairobi", "Nairobi"),     # start and destination are the same
    ("Nairobi", "Narnia"),      # unknown town
    ("", "Kisumu"),             # empty input
    ("Nairobi", "Mombasa"),     # known towns but no road links them
]

print("\n--- Debugging the Knowledge Base (route queries) ---")
for s, g in test_cases:
    print(f"Query: '{s}' → '{g}'")
    print_route(find_route(s, g, debug_kb))
    print()

print("--- Debugging the Knowledge Base (faulty new facts) ---")
bad_facts = [("Nairobi", "Thika", -5), ("Nairobi", "Thika", "far"), ("Nakuru", "Nakuru", 10), ("", "Nakuru", 10)]
for a, b, km in bad_facts:
    try:
        add_new_fact(a, b, km, debug_kb)
    except ValueError as err:
        print(f"Rejected Road('{a}', '{b}', {km!r}): {err}")

Added fact: Road(Mombasa, Voi, 150) and Road(Voi, Mombasa, 150)

--- Debugging the Knowledge Base (route queries) ---
Query: 'Nairobi' → 'Kisumu'
Agent decision: take Nairobi → Naivasha → Nakuru → Kericho → Kisumu
Total distance: 365 km

Query: 'nairobi' → '  KISUMU '
Agent decision: take Nairobi → Naivasha → Nakuru → Kericho → Kisumu
Total distance: 365 km

Query: 'muranga' → 'nyeri'
Agent decision: take Murang'a → Nyeri
Total distance: 55 km

Query: 'Nairobi' → 'Nairobi'
Agent decision: already at Nairobi (0 km).

Query: 'Nairobi' → 'Narnia'
Agent decision: Unknown town: 'Narnia' is not in the knowledge base.

Query: '' → 'Kisumu'
Agent decision: No town name was given.

Query: 'Nairobi' → 'Mombasa'
Agent decision: No known route from Nairobi to Mombasa in the knowledge base.

--- Debugging the Knowledge Base (faulty new facts) ---
Rejected Road('Nairobi', 'Thika', -5): Distance must be greater than zero.
Rejected Road('Nairobi', 'Thika', 'far'): Distance must be a number.
Rejected R

You can then add or edit facts based on the test cases, for example add the missing road between two towns and ask for the route again.

**Step 10: Final Reflection** 

**Challenges**

- **Abstraction:** deciding what to keep from the real road network (towns, links, distances) and what to leave out (traffic, weather, road quality, elevation).
- **Representation:** storing each road once but making it usable in both directions, which is why the two-way rule is applied automatically in `tell_road`.
- **Messy input:** users may type different capitalisation, drop the apostrophe in Murang'a, enter unknown towns or give invalid distances, so every query and new fact is validated.
- **Approximate data:** the distances are approximate and fixed, so the agent's answers are only as good as the facts it is given.

**Recommended improvements**

- Retrieve live distances or travel times from a maps API (where permitted) instead of typing them in by hand.
- Add costs beyond distance, such as fuel, tolls, weighbridge stops, road condition and cargo weight.
- Use A* with a straight-line-distance heuristic to search faster on a larger network.
- Use a logic engine (for example Prolog or a Python Datalog library) so rules can be written declaratively rather than hard-coded.
- Save the knowledge base to a file so facts added by users are kept between runs.